# Topology-weight warm-up experiment (Colab)

**Question.** The topology loss made the models over-draw vessels at the fixed 0.5 threshold, which cost Dice (stage C vs B). Does *warming up* the topology
weight (0 -> full over the first 2000 of 5000 iterations, so Dice/BCE shape the probabilities first) remove that cost?

**What this notebook runs** (one pre-registered setting, not tuned; checkpoints are still chosen on DRIVE-validation only):

| experiment | recipe | seeds |
|---|---|---|
| `P_unet_topo_dg_warm` | stage C + warm-up | 0, 1, 2 |
| `P_lite_topo_dg_warm` | stage D + warm-up | 0, 1, 2 |

Compare with `C_unet_topo_dg` and `D_lite_topo_dg` (same seeds 0-2, already in the repository).

**Before you start**
1. *Runtime -> Change runtime type -> GPU* (T4 is fine; L4/A100 are faster).
2. The code with the warm-up option must already be pushed to GitHub (`git push` on your laptop).
3. Expect about 3-5 hours in total (U-Net runs ~40 min each, compact runs ~15 min each, plus evaluation). Keep the tab open; the runner skips finished
   runs, so if the session dies just run the cells again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
NW = min(4, os.cpu_count() or 2)          # data-loader / metric workers
print("CPU cores:", os.cpu_count(), "-> using", NW, "workers")

## 1. Get the code and the data

In [ ]:
!git clone https://github.com/Deltasthicc/ML-Project.git
%cd ML-Project
!pip install -q -r requirements.txt
!pip install -q -e .
!python -c "import tglite.experiments as e; print(e.GROUPS['warmup'])"

In [ ]:
# downloads DRIVE, STARE, CHASE_DB1 and HRF (a few minutes) and prepares them; every dataset must print OK
!python scripts/download_data.py
!python scripts/prepare_data.py

## 2. Smoke test (about 1 minute)

In [ ]:
!python scripts/run_experiments.py --exp P_lite_topo_dg_warm --seeds 0 --quick

## 3. Train and evaluate (the long step)
Writes to `results/runs/P_*/seed*/`. Safe to re-run after an interruption.

In [ ]:
!python -u scripts/run_experiments.py --groups warmup --seeds 0 1 2 --time-budget-hours 6 --set train.num_workers={NW} eval.metric_workers={NW} 2>&1 | tee warmup_log.txt

## 4. Fragment clean-up and the validation-tuned threshold (evaluation only, about 40 minutes)

In [ ]:
!python scripts/eval_postproc.py --runs results/runs --exp P_unet_topo_dg_warm P_lite_topo_dg_warm --seeds 0 1 2 --workers {NW}

## 5. Quick look

In [ ]:
import pandas as pd
!python scripts/aggregate_results.py
t = pd.read_csv("results/tables/summary.csv")
rows = []
for e in ["C_unet_topo_dg", "P_unet_topo_dg_warm", "D_lite_topo_dg", "P_lite_topo_dg_warm"]:
    r = {"exp": e}
    for m in ("dice", "cldice", "betti0_err"):
        q = t[(t.exp == e) & (t.dataset == "unseen") & (t.metric == m) & (t["mode"] == "det")]
        r[m] = round(float(q["mean"].iloc[0]), 4) if len(q) else None
    rows.append(r)
pd.DataFrame(rows)

## 6. Download the results (no checkpoints, a few MB)

In [ ]:
!zip -qr warmup_results.zip results/runs/P_unet_topo_dg_warm results/runs/P_lite_topo_dg_warm warmup_log.txt -x "*.pt" "*/preds/*"
from google.colab import files
files.download("warmup_results.zip")

## 7. Back on the laptop

```bash
# in the ML-Project folder: unzip so that results/runs/P_* appear, then
python scripts/aggregate_results.py
python scripts/eval_postproc.py --runs results/runs --no-eval --exp A_unet B_unet_topo C_unet_topo_dg D_lite_topo_dg E_tglitenet X_unet_dg S_unet_b8 S_unet_b16 X_lite_dg P_unet_topo_dg_warm P_lite_topo_dg_warm --seeds 0 1 2 3 4 --out results/tables
```
Then read `results/tables/stage_deltas.csv`-style paired differences in `factorial_deltas.csv` (C vs P_unet..., D vs P_lite...) and `postproc_deltas.csv`.
Report the result whichever way it goes.